In [0]:
# =============================================================================
# NOTEBOOK  : bronze_warehouse_inventory
# PURPOSE   : Ingest warehouse_inventory.parquet → bronze.warehouse_inventory
# LAYER     : Bronze (raw ingestion — no transformation)
# FREQUENCY : Hourly (Autoloader availableNow)
# FORMAT    : Parquet
# NOTE      : Stock columns arrive as LONG from source — matched in schema
#             last_updated arrives as STRING from source — cast in silver
#             available_stock kept as-is from source — recomputed in silver
# =============================================================================

# ── Imports & Config ──────────────────────────────────────────────────
import sys, os, json
from pathlib import Path

# Derive project root dynamically from notebook path. mini_project_a3t7 is always the git folder name
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"

sys.path.insert(0, PROJECT_ROOT)

from utils.audit import start_audit, end_audit
from utils.quality_checks import assert_not_empty
from utils.schema_registry import BRONZE_WAREHOUSE_INVENTORY_SCHEMA

from pyspark.sql.functions import current_timestamp, lit, col, to_date

with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)

SOURCE_PATH  = cfg["landing_paths"]["warehouse_inventory"]
TARGET_TABLE = cfg["tables"]["bronze_warehouse_inventory"]
CHECKPOINT   = cfg["checkpoint_paths"]["bronze_warehouse_inventory"]
PIPELINE     = "bronze_warehouse_inventory"

In [0]:
# ── Audit + Read + Write ─────────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE)

try:
    # ── READ (Autoloader — availableNow) ─────────────────────────────────────
    # Parquet is typed — schema must match exactly what source sends
    # If source changes schema, job fails loudly (mergeSchema not set)
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", CHECKPOINT + "/schema")
        .schema(BRONZE_WAREHOUSE_INVENTORY_SCHEMA)
        .load(SOURCE_PATH)
    )

    # ── ADD AUDIT COLUMNS ─────────────────────────────────────────────────────
    df = (
        df
        .withColumn("source_file",     col("_metadata.file_path"))
        .withColumn("ingested_at",     current_timestamp())
        .withColumn("ingested_date",   to_date(current_timestamp()).cast("string"))
        .withColumn("pipeline_run_id", lit(run_id))
    )

    # ── WRITE ─────────────────────────────────────────────────────────────────
    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT)
        .partitionBy("ingested_date")
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
    )

    query.awaitTermination()

    # ── QUALITY CHECK ─────────────────────────────────────────────────────────
    written_df = (
        spark.read.table(TARGET_TABLE)
        .where(f"pipeline_run_id = '{run_id}'")
    )
    rows_written = written_df.count()
    assert_not_empty(written_df, PIPELINE)

    print(f"[WRITE] {rows_written} rows written to {TARGET_TABLE}")

    # ── END AUDIT (SUCCESS) ───────────────────────────────────────────────────
    end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS", rows_written=rows_written)

except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise